# 03. Evaluation và xếp hạng OCR

Chọn kernel `.venv-ocr`. So sánh 9 hệ thống: VietLPR_v1, VietLPR_v2 và 7 bản PP-OCRv4/v5/v6.
Mặc định đọc kết quả đã lưu. Bật `RUN_EVALUATION` khi cần chuẩn bị/tiếp tục đo.
Đầu vào là ảnh biển; toàn bộ DET → crop → REC chạy trên CPU ONNX FP32.

In [1]:
from pathlib import Path
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "ocr/config.py").is_file() and (p / "configs/ocr").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Mở notebook trong thư mục dự án vn-lpr-pipeline.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
PYTHON = ROOT / ".venv-ocr/Scripts/python.exe"
if Path(sys.executable).resolve() != PYTHON.resolve():
    raise RuntimeError(f"Chọn kernel .venv-ocr: {PYTHON}")
from ocr.config import load_config
from ocr.notebook import run, require_files
print("Workspace:", ROOT)

Workspace: D:\DUT\NCKH\dataset


## 1. Cấu hình evaluation

`RECALIBRATE=True` để chốt lại hậu xử lý trên val trước test; không chọn theo kết quả test.
Paddle giữ ngưỡng nguồn. Để dò lưới val, chạy CLI `ocr.paddle_suite tune --search` trước bước test.

In [2]:
RUN_EVALUATION = False
RECALIBRATE = False
PADDLE_PY = ROOT / ".venv-ocr-paddle/Scripts/python.exe"
VIETLPR_MODELS = ("vietlpr_v1", "vietlpr_v2")
PADDLE_TEST_ORDER = (
    "PP-OCRv4_mobile", "PP-OCRv5_mobile", "PP-OCRv6_tiny", "PP-OCRv6_small",
    "PP-OCRv6_medium", "PP-OCRv5_server", "PP-OCRv4_server",
)

## 2. Chuẩn bị model và chốt val

VietLPR cần best ONNX từ notebook 02. Paddle dùng pretrained chính thức: v4 chuyển bằng môi trường Paddle riêng;
v5/v6 tải ONNX chính thức. Kiểm checksum/dictionary trước khi đo, không train Paddle.

In [3]:
if RUN_EVALUATION:
    run(PYTHON, "-m", "ocr.paddle_suite", "prepare", "--paddle-python", PADDLE_PY)
    for name in VIETLPR_MODELS:
        directory = Path(load_config(name)["run_dir"])
        require_files([directory / "onnx" / f for f in ("det.onnx", "rec.onnx", "export_check.json")], "Xuất ONNX tại notebook 02 trước.")
        if RECALIBRATE or not (directory / "onnx/frozen.json").is_file():
            run(PYTHON, "-m", "ocr.evaluate", "tune", "--model", name)
    for name in PADDLE_TEST_ORDER:
        if RECALIBRATE or not (ROOT / "runs/paddle_suite" / name / "onnx/frozen.json").is_file():
            run(PYTHON, "-m", "ocr.paddle_suite", "tune", "--model", name)
else:
    print("Đọc kết quả đã lưu; chưa chạy lại evaluation.")

Đọc kết quả đã lưu; chưa chạy lại evaluation.


## 3. Đo trên tập test

CPU 4 luồng, OpenCV 1 luồng, FP32, batch 1 ảnh biển, warm-up 20 và 5 lượt trên đủ 124 ảnh test.
Speed gồm tiền xử lý, DET, hậu xử lý, crop, REC và decode; không tính đọc ổ đĩa hoặc load model.
Paddle lưu sau từng ảnh vào `benchmark_progress.json`; chạy lại sẽ tiếp tục phần còn thiếu.
Đã đủ 5 lượt thì dùng lại số đo. Muốn đo mới, cất tiến độ và kết quả lượt cũ trước khi chạy lại.

In [4]:
if RUN_EVALUATION:
    for name in VIETLPR_MODELS:
        run(PYTHON, "-m", "ocr.evaluate", "test", "--model", name)
    for name in PADDLE_TEST_ORDER:
        run(PYTHON, "-m", "ocr.paddle_suite", "test", "--model", name)
else:
    print("RUN_EVALUATION=False; dùng các số đo đã lưu.")

RUN_EVALUATION=False; dùng các số đo đã lưu.


## 4. Bảng xếp hạng

Rank: ACC giảm dần → CER tăng dần → Speed tăng dần. **Speedup lấy model chậm nhất = 1×**.
VietLPR đã train trên dataset v2; Paddle dùng pretrained. REC V2 gom các dòng trong cùng ảnh, các hệ thống còn lại chạy từng dòng.
Kết quả được đo ở các thời điểm khác nhau trên cùng CPU/runtime và tập test; Speed có dao động giữa các lượt.
`runs/comparison_ranked.csv` lưu bảng đầy đủ. Các chỉ số gốc và từng dự đoán nằm trong `runs/<model>/`.

In [5]:
from IPython.display import Markdown, display
from ocr.comparison import collect_results, rank_results, save_table, markdown_table

rows = rank_results(collect_results())
table = save_table(rows)
display(Markdown(markdown_table(rows)))
print("CSV:", table)

CSV: D:\DUT\NCKH\dataset\runs\comparison_ranked.csv


| Rank | Model | ACC (%) ↑ | CER (%) ↓ | Speed (ms/ảnh) ↓ | FPS ↑ | Speedup (×) ↑ |
|---:|---|---:|---:|---:|---:|---:|
| 1 | VietLPR_v2 | 94.35 | 0.68 | 1.51 | 663.82 | 4194.24 |
| 2 | PP-OCRv5_server | 93.55 | 0.77 | 2242.14 | 0.45 | 2.82 |
| 3 | PP-OCRv4_mobile | 91.94 | 1.93 | 38.52 | 25.96 | 164.03 |
| 4 | PP-OCRv6_medium | 87.90 | 2.31 | 187.92 | 5.32 | 33.62 |
| 5 | PP-OCRv4_server | 87.90 | 6.75 | 6318.31 | 0.16 | 1.00 |
| 6 | PP-OCRv6_small | 87.10 | 4.82 | 24.58 | 40.68 | 257.02 |
| 7 | PP-OCRv5_mobile | 69.35 | 4.34 | 31.85 | 31.40 | 198.38 |
| 8 | PP-OCRv6_tiny | 58.06 | 14.75 | 8.54 | 117.06 | 739.63 |
| 9 | VietLPR_v1 | 26.61 | 42.33 | 19.30 | 51.82 | 327.40 |